This notebook is used to extract the residual relief (RR) from topography, calculate volumetric obscurance (VO) from RR, crop the VO by a dune mask and perform a top-down skeletonization process to generate dune crestlines as vectors following Daynac et al. 2024.

### Imports

In [ ]:
import rasterio
import rasterio.mask
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window, from_bounds
from rasterio.transform import rowcol

from shapely import get_point, overlaps, line_merge

import geopandas as gpd

import glob
import matplotlib.pyplot as plt
import numpy as np
import fiona
from scipy.ndimage import median_filter
import time
from skimage.morphology import thin, skeletonize

import os

### Functions

In [ ]:
### saving a tif definitions ###
def save_tif(img, transform, crs, filename, nodata=np.nan):
    """save an image using rasterio"""
    height, width = img.shape
    with rasterio.open(
        filename,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=img.dtype,
        crs=crs,
        transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(img, 1)

### VO related definitions ###
# this code is from (with small adjustments): Rolland T, Monna F, Buoncristiani JF, et al (2022) Volumetric Obscurance as a New Tool to Better Visualize Relief from Digital Elevation Models. Remote Sensing 14:941. https://doi.org/10.3390/rs14040941
class Calculation():

    def __init__(self, grid, d, h_vert_cor,
                 image_final_npy):
        
        super(Calculation, self).__init__()
        
        self.h_vert_cor = h_vert_cor
        self.d = d
        self.grid = grid
        self.image_final_npy = image_final_npy

    def run_on_cpu(self):                
        # construction of empty arrays to receive the results of treatments
        self.large_vo = np.zeros((self.image_final_npy.shape[0], self.image_final_npy.shape[1], 2))   
        
        for i in range(0, self.d.size):
            # print("%d/%d"%(i,self.d.size),end="\r") # UNCOMMENT TO HAVE A BETTER IDEA OF WHERE THE CODE IS IF YOU LIKE
            M = (np.roll(self.image_final_npy, [self.grid[i, 0], self.grid[i, 1]], axis = [0, 1]) - self.image_final_npy)
            vop = self.h_vert_cor[i] - M                       
            vop[vop > 2 * self.h_vert_cor[i]] = 2 * self.h_vert_cor[i]
            vop[vop < 0] = 0
            self.large_vo[:,:,1] = vop
            self.large_vo[:,:,0]  = np.sum(self.large_vo, axis=2)  #cumulative sum
            
def get_VO(img,radius,exageration=1000):
    x = np.arange(-radius, radius + 1, dtype = int)  
    y = np.arange(-radius, radius + 1, dtype = int)  
    grid = np.stack(np.meshgrid(x, y), -1).reshape(-1, 2) 
    distance = np.sqrt(grid[:,0]**2 + grid[:,1]**2) 
    d = distance[distance < radius] 
    grid = grid[distance < radius]   
    h_vert_cor = np.sqrt(radius**2 - d**2) * 30 / exageration # 30 meter was chosen as this is roughly the resolution of COP30 (although that obviously depends on the lat)
    calc = Calculation(grid, d, h_vert_cor, img)
    calc.run_on_cpu()
    out_vo = calc.large_vo[:,:,0]  / (2 * h_vert_cor.sum())
    out_vo = median_filter(out_vo,3) # smoothing to reduce noise
    return out_vo

### Skeleton related definitions ###
# code from (and slightly adapted): Tveite H (2017) The QGIS ThinGreyscale Plugin [software]. https://github.com/havatv/qgisthingreyscaleplugin, accessed: 30 July 2025
def skeletonize_raster(inputlayer,levels):
    # Create a numpy array version of the image with zeros (by me)
    imageMat = np.where(np.isnan(inputlayer),0,inputlayer)
    height = imageMat.shape[0]
    width = imageMat.shape[1]
    
    # Initialise the skeleton raster as a matrix of boolean
    # Add an "edge" around the image to protect the filter
    SkelMat = np.zeros((height + 2, width + 2), dtype=np.uint8)

    minrastervalue = levels[0]
    maxrastervalue = levels[-1]
    # remove last Item of levels list
    levels.pop()

    # Go through the levels (start at the top)
    for level in reversed(levels):
        if level < minrastervalue:
            break
        ThresMat = np.zeros((height + 2, width + 2), dtype=bool)
        relevantpixels = np.where((imageMat >= level) &
                           (imageMat <= maxrastervalue))
        checkpixels = zip(relevantpixels[0], relevantpixels[1])
        for (row, column) in checkpixels:
            ThresMat[row + 1, column + 1] = True
        SkelMat = np.add(ThresMat, SkelMat)
        changes = True

        # Step 1
        while changes:
            changes = False

            newSkelMat = np.array(SkelMat, dtype=np.uint8)
            
            relevantpixels = np.where(SkelMat == 1)
            checkpixels = zip(relevantpixels[0], relevantpixels[1])
            for (row, column) in checkpixels:
                B = 0
                A = 0
                #P1 = SkelMat[row,column]
                P2 = SkelMat[row - 1, column]
                P3 = SkelMat[row - 1, column + 1]
                P4 = SkelMat[row, column + 1]
                P5 = SkelMat[row + 1, column + 1]
                P6 = SkelMat[row + 1, column]
                P7 = SkelMat[row + 1, column - 1]
                P8 = SkelMat[row, column - 1]
                P9 = SkelMat[row - 1, column - 1]
                if P2 > 0:
                    B = B + 1
                    if P9 == 0:
                        A = A + 1
                if P3 > 0:
                    B = B + 1
                    if P2 == 0:
                        A = A + 1
                if P4 > 0:
                    B = B + 1
                    if P3 == 0:
                        A = A + 1
                if P5 > 0:
                    B = B + 1
                    if P4 == 0:
                        A = A + 1
                if P6 > 0:
                    B = B + 1
                    if P5 == 0:
                        A = A + 1
                if P7 > 0:
                    B = B + 1
                    if P6 == 0:
                        A = A + 1
                if P8 > 0:
                    B = B + 1
                    if P7 == 0:
                        A = A + 1
                if P9 > 0:
                    B = B + 1
                    if P8 == 0:
                        A = A + 1
                tozero = (B >= 2 and B <= 6 and A == 1 and
                              (P2 == 0 or P4 == 0 or P6 == 0) and
                              (P4 == 0 or P6 == 0 or P8 == 0))

                if tozero:
                    changes = True
                    newSkelMat[row, column] = 0
            SkelMat = newSkelMat
            
            newSkelMat = np.array(SkelMat, dtype=np.uint8)
            relevantpixels = np.where(SkelMat == 1)
            checkpixels = zip(relevantpixels[0], relevantpixels[1])
            for (row, column) in checkpixels:
                #P1 = SkelMat[row,column]
                B = 0
                A = 0
                P9 = SkelMat[row - 1, column - 1]
                P2 = SkelMat[row - 1, column]
                P3 = SkelMat[row - 1, column + 1]
                P4 = SkelMat[row, column + 1]
                P5 = SkelMat[row + 1, column + 1]
                P6 = SkelMat[row + 1, column]
                P7 = SkelMat[row + 1, column - 1]
                P8 = SkelMat[row, column - 1]
                if P2 > 0:
                    B = B + 1
                    if P9 == 0:
                        A = A + 1
                if P3 > 0:
                    B = B + 1
                    if P2 == 0:
                        A = A + 1
                if P4 > 0:
                    B = B + 1
                    if P3 == 0:
                        A = A + 1
                if P5 > 0:
                    B = B + 1
                    if P4 == 0:
                        A = A + 1
                if P6 > 0:
                    B = B + 1
                    if P5 == 0:
                        A = A + 1
                if P7 > 0:
                    B = B + 1
                    if P6 == 0:
                        A = A + 1
                if P8 > 0:
                    B = B + 1
                    if P7 == 0:
                        A = A + 1
                if P9 > 0:
                    B = B + 1
                    if P8 == 0:
                        A = A + 1
                tozero = (B >= 2 and B <= 6 and A == 1 and
                          (P2 == 0 or P4 == 0 or P8 == 0) and
                          (P2 == 0 or P6 == 0 or P8 == 0))
                if tozero:
                    changes = True
                    newSkelMat[row, column] = 0
            SkelMat = newSkelMat

        ## Step 2
        newSkelMat = np.array(SkelMat, dtype=np.uint8)
        relevantpixels = np.where(SkelMat == 1)
        checkpixels = zip(relevantpixels[0], relevantpixels[1])
        for (row, column) in checkpixels:
            #P1 = SkelMat[row, column]
            B = 0
            P2 = SkelMat[row - 1, column]
            P3 = SkelMat[row - 1, column + 1]
            P4 = SkelMat[row, column + 1]
            P5 = SkelMat[row + 1, column + 1]
            P6 = SkelMat[row + 1, column]
            P7 = SkelMat[row + 1, column - 1]
            P8 = SkelMat[row, column - 1]
            P9 = SkelMat[row - 1, column - 1]
            if P2:
                B = B + 1
            if P3:
                B = B + 1
            if P4:
                B = B + 1
            if P5:
                B = B + 1
            if P6:
                B = B + 1
            if P7:
                B = B + 1
            if P8:
                B = B + 1
            if P9:
                B = B + 1
            tozero = (B >= 7)
            #if tozero or fourer:
            if tozero:
                newSkelMat[row, column] = 0
        SkelMat = newSkelMat
        
        # print to console
        print("level %2f"%(level),end="\r")

    return SkelMat[1:-1,1:-1].astype(np.uint8)

def get_middle_image(left_file, middle_file, right_file, radius_half):
    
    # Read full middle file
    with rasterio.open(middle_file) as srcm:
        
        # Read window from left file
        try:
            with rasterio.open(left_file) as srcl:
                window_left = Window(srcl.shape[1] - radius_half, 0, radius_half, srcl.shape[0])
                img_left = srcl.read(1, window=window_left)
        except TypeError:
            img_left=np.full((srcm.shape[0],radius_half),np.nan)
    
        # Read window from right file
        try:
            with rasterio.open(right_file) as srcr:
                window_right = Window(0, 0, radius_half, srcr.shape[0])
                img_right = srcr.read(1, window=window_right)
        except TypeError:
            img_right=np.full((srcm.shape[0],radius_half),np.nan)

        # read data
        img_middle = srcm.read(1)
        
        # pad the middle image with half the radius data fromtif left and right of it, pad the top and bottom
        img_middle = np.concatenate([img_left,img_middle,img_right],axis=1)
        img_middle = np.pad(img_middle, ((radius_half, radius_half), (0, 0)), mode='constant', constant_values=0)
        
        return img_middle, srcm.transform, srcm.crs

### Prepare mask

Adjust original mask provided by Daynac by cropping it and reprojecting it to the extent of topography data. Also store as uint8 to save storage instead of float32

In [ ]:
save=False 

# paths to all Copernicus files of interest
small_tif_paths = sorted(list(glob.glob("../Data/Copernicus_OpenTopo_Australia/*tif")))

# load the first (western most) raster to get transform for Mask
with rasterio.open(small_tif_paths[0]) as small_raster:
    small_transform = small_raster.transform
    small_width = small_raster.width
    small_height = small_raster.height
    small_crs = small_raster.crs
    pixel_size_x = small_transform.a
    pixel_size_y = -small_transform.e

# Define the number of small TIFFs in the x and y directions (grid layout)
num_small_x = len(small_tif_paths)  # we only have tiffs that extend in x
num_small_y = 1                     # all tiffs have the same y extent

# Calculate the combined dimensions and transform (as if all tifs were stiched together)
combined_width = small_width * num_small_x
combined_height = small_height * num_small_y

# Calculate the combined transform based on the top-left corner of the first small raster and the extent of all tifs combined
combined_transform = rasterio.transform.from_origin(
    small_transform.c,  # Top-left x coordinate
    small_transform.f,  # Top-left y coordinate
    pixel_size_x,       # Pixel width
    pixel_size_y        # Pixel height
)

# align the larger raster to the combined grid of the smaller tifs to make mask align with the topography
with rasterio.open('../Data/Daynac_dune_mask/00_Veg_linear_dune_AUSTRALIA.tif') as large_raster:
    large_data = large_raster.read(1).astype(np.uint8)  # original Daynac mask is float32 but uint8 is sufficient
    large_transform = large_raster.transform
    large_crs = large_raster.crs

    # prepare an array for the aligned data
    aligned_large_data = np.empty((combined_height, combined_width), dtype=large_data.dtype)

    # reproject the large raster to align with the combined grid
    reproject(
        source=large_data,
        destination=aligned_large_data,
        src_transform=large_transform,
        src_crs=large_crs,
        dst_transform=combined_transform,
        dst_crs=small_crs,
        resampling=Resampling.nearest
    )

# save the reprojected Mask to use in on the VO
if save:
    save_tif(aligned_large_data,combined_transform,small_crs,"../Data/Daynac_dune_mask/Dune_mask_Australia.tif",nodata=None)

### Calculate the RR per COP tif

Calculating the Resudial Relief (RR) from topography follwing Daynac et al. 2024

In [ ]:
start = time.time()

save=False   # change to True to actually save
radius = 20  # pixels following Daynac et al. 2024
radius_half = int(radius/2)
cop_files = sorted(list(glob.glob("../Data/Copernicus_OpenTopo_Australia/*.tif")))[:-4] # no need for the last 4 since there are no dunes

for i in range(len(cop_files)):
    if cop_files[i] == cop_files[0]:
        #handle leftmost case
        left_file = None
        middle_file = cop_files[i]
        right_file = cop_files[i+1]
        
    elif cop_files[i] == cop_files[-1]:
        # handle rightmost case
        left_file = cop_files[i-1]
        middle_file = cop_files[i]
        right_file = None
    else:
        # handle middle cases
        left_file = cop_files[i-1]
        middle_file = cop_files[i]
        right_file = cop_files[i+1]
        
    img_middle, middle_transform, middle_crs = get_middle_image(left_file,middle_file,right_file,radius_half)

    if save:
        output_file = "../Data/RR/RR%s.tif"%(middle_file.split("\\")[-1][8:-4]) # \\ or / depending on OS
        if not os.path.isfile(output_file):
            # compute the RR
            print("Calculating RR for %s"%(output_file))
            img_middle = img_middle-median_filter(img_middle,radius)
            save_tif(img_middle[radius_half:-radius_half,radius_half:-radius_half].astype(np.float32),
                     middle_transform,
                     middle_crs,
                     output_file,
                     nodata=np.nan)
            print("Calculated RR in %05ds"%(time.time()-start))
        else:
            print("File already exists: %s"%(output_file))

### Calculate the VO per RR tif

Calculate the volumetric obscruance (VO) from the RR following Daynac et al. 2024 and Rolland et al. 2022

In [ ]:
start = time.time()

save=False   # adjust to actually save
radius = 20  # pixels
radius_half = int(radius/2)
RR_files = sorted(list(glob.glob("../Data/RR/*.tif")))

for i in range(len(RR_files)):
    # if not RR_files[i] == RR_files[-6]:
    #     continue
    if RR_files[i] == RR_files[0]:
        #handle leftmost case
        left_file = None
        middle_file = RR_files[i]
        right_file = RR_files[i+1]
        
    elif RR_files[i] == RR_files[-1]:
        # handle rightmost case
        left_file = RR_files[i-1]
        middle_file = RR_files[i]
        right_file = None
    else:
        # handle middle cases
        left_file = RR_files[i-1]
        middle_file = RR_files[i]
        right_file = RR_files[i+1]
        
    img_middle, middle_transform, middle_crs = get_middle_image(left_file,middle_file,right_file,radius_half)

    if save:
        output_file = "../Data/VO/VO%s.tif"%(middle_file.split("\\")[-1][2:-4]) # \\ or / depending on OS
        if not os.path.isfile(output_file):
            # compute the VO
            print("Calculating VO for %s"%(output_file))
            img_middle=get_VO(img_middle,radius_half)            
            save_tif(img_middle[radius_half:-radius_half,radius_half:-radius_half].astype(np.float32),
                     middle_transform,
                     middle_crs,
                     output_file,
                     nodata=np.nan)
            print("Calculated VO in %05ds"%(time.time()-start))
        else:
            print("File already exists: %s"%(output_file))

### Calculate the Skeleton per VO tif

Caculate the Skeleton (centre line) of dunes from the VO using the dune mask from Daynac et al. 2024 and code from Tveite 2017 (https://github.com/havatv/qgisthingreyscaleplugin, accessed: 30 July 2025)

In [ ]:
save=False   # adjust to save
radius = 20  # pixels
radius_half = int(radius/2)
VO_files = sorted(list(glob.glob("../Data/VO/*.tif")))

for i in range(len(VO_files)):
    start = time.time()
    # if not VO_files[i] == VO_files[0]:
    #     continue
    if VO_files[i] == VO_files[0]:
        #handle leftmost case
        left_file = None
        middle_file = VO_files[i]
        right_file = VO_files[i+1]
        
    elif VO_files[i] == VO_files[-1]:
        # handle rightmost case
        left_file = VO_files[i-1]
        middle_file = VO_files[i]
        right_file = None
    else:
        # handle middle cases
        left_file = VO_files[i-1]
        middle_file = VO_files[i]
        right_file = VO_files[i+1]
        
    img_middle, middle_transform, middle_crs = get_middle_image(left_file,middle_file,right_file,radius_half)
    
    # dune mask
    with rasterio.open("../Data/Daynac_dune_mask/Dune_mask_Australia.tif") as src:
        wnd_row_off_mask, wnd_col_off_mask = src.index(middle_transform.c,middle_transform.f)
        if left_file is None:
            window_mask=Window(wnd_col_off_mask,wnd_row_off_mask,img_middle.shape[1]-radius_half,img_middle.shape[0]-radius)
            img_mask = src.read(1,window=window_mask)
            img_mask = np.pad(img_mask, ((radius_half, radius_half), (radius_half, 0)), mode='constant', constant_values=0)
        elif right_file is None:
            window_mask=Window(wnd_col_off_mask-radius_half,wnd_row_off_mask,img_middle.shape[1]-radius_half,img_middle.shape[0]-radius)
            img_mask = src.read(1,window=window_mask)
            img_mask = np.pad(img_mask, ((radius_half, radius_half), (0, radius_half)), mode='constant', constant_values=0)
        else:    
            window_mask=Window(wnd_col_off_mask-radius_half,wnd_row_off_mask,img_middle.shape[1],img_middle.shape[0]-radius)
            img_mask = src.read(1,window=window_mask)
            img_mask = np.pad(img_mask, ((radius_half, radius_half), (0, 0)), mode='constant', constant_values=0)

    if save:
        output_file = "../Data/SKL/SKL%s.tif"%(middle_file.split("\\")[-1][2:-4])  # \\ or / depending on OS
        if not os.path.isfile(output_file):
            # compute the VO
            print("Calculating SKL for %s"%(output_file))
            # crop the VO by the dune mask and create skeleton
            img_middle = np.where(img_mask,img_middle,np.nan)
            #skeletonization levels needs to be resseted everytime (just ot be safe)
            levels = list(np.linspace(0.1,1,10))
            # innermost skeletonization is top-down skeletonization, then thinning, and then skeletonize using "lee" to mimic the r.thin
            img_middle = skeletonize(thin(skeletonize_raster(img_middle,levels)).astype(np.uint8),method="lee").astype(np.uint8)
            save_tif(img_middle[radius_half:-radius_half,radius_half:-radius_half],
                     middle_transform,
                     middle_crs,
                     output_file,
                     nodata=0)
            print("Calculated SKL in %05ds"%(time.time()-start))
        else:
            print("File already exists: %s"%(output_file))

### Merge all Skeleton raster into one combined Raster

Combine the centre line Skeleton (raster) to one continous raster for line extraction (vectorization)

In [ ]:
start = time.time()

SKL_files = sorted(list(glob.glob("../Data/SKL/*.tif")))
datasets = [rasterio.open(file) for file in SKL_files]
merged_data, merged_transform = merge(datasets)

# Use the metadata from one of the source datasets
output_meta = datasets[0].meta.copy()

# Update metadata to reflect the new shape, transform, and number of layers
output_meta.update({
    "driver": "GTiff",
    "height": merged_data.shape[1],
    "width": merged_data.shape[2],
    "transform": merged_transform
})

# Save the merged raster
with rasterio.open("../Data/Crestlines/SKL_combined.tif", "w", **output_meta) as dest:
    dest.write(merged_data)

print(time.time()-start)

### Steps in QGIS (can take some time and make QGIS not respond for a while)

To turn the combined Skeleton raster "SKL_combined" into vector lines we need to use some QGIS plugins (following Daynac et al. 2024):
1. Use the "r.thin" toolbox to thin the raster
2. Use the "r.to.vector" toolbox to turn the thinned raster into a vector (type = line)
3. Use the "v.generalize" on this new vector feature to smooth the crest lines and save as "SKL_combined.shp" (algorithm = snake)